In [0]:

from pyspark.sql import functions as F

# Read our enriched Silver Delta table
df_silver = spark.table(
    "workspace.silver.citibike_station_enriched"
)

# Find the most recent ingestion timestamp
latest_timestamp = df_silver.agg(
    F.max("ingestion_timestamp")
).first()[0]

# Keep records from the latest snapshot only
df_latest = df_silver.filter(
    F.col("ingestion_timestamp") == latest_timestamp
)

# Display the results
print("Total Silver records:", df_silver.count())
print("Latest snapshot timestamp:", latest_timestamp)
print("Latest snapshot records:", df_latest.count())

display(
    df_latest.select(
        "station_name",
        "bikes_available",
        "docks_available",
        "capacity",
        "ingestion_timestamp"
    ).limit(5)
)


Total Silver records: 30228
Latest snapshot timestamp: 2026-11-24 22:37:32.875862
Latest snapshot records: 2519


station_name,bikes_available,docks_available,capacity,ingestion_timestamp
W 35 St & 8 Ave,0,0,45,2026-11-24T22:37:32.875Z
7 St & 5 Ave,0,0,0,2026-11-24T22:37:32.875Z
Broadway & W 41 St,0,0,55,2026-11-24T22:37:32.875Z
Prospect Pl & Nostrand Ave,0,0,0,2026-11-24T22:37:32.875Z
51 Ave & 61 St,0,0,0,2026-11-24T22:37:32.875Z


In [0]:

from pyspark.sql import functions as F

# Read Silver data
df_silver = spark.table(
    "workspace.silver.citibike_station_enriched"
)

# Find the latest ingestion timestamp
latest_timestamp = df_silver.agg(
    F.max("ingestion_timestamp")
).first()[0]

# Stop if no data exists
if latest_timestamp is None:
    raise ValueError("Silver table has no ingestion timestamps!")

# Calculate data age using Spark
data_age_minutes = spark.sql(f"""
    SELECT (
        unix_timestamp(current_timestamp()) -
        unix_timestamp(TIMESTAMP '{latest_timestamp}')
    ) / 60.0 AS data_age_minutes
""").first()["data_age_minutes"]

print("Latest Silver ingestion:", latest_timestamp)
print("Data age (minutes):", round(data_age_minutes, 2))

# Check freshness
if data_age_minutes > 45:
    raise ValueError(
        f"Silver data is stale: {data_age_minutes:.2f} minutes old!"
    )

if data_age_minutes < -5:
    raise ValueError("Silver timestamp is in the future!")

print("SUCCESS: Silver data freshness validation passed!")


Latest Silver ingestion: 2026-11-24 22:37:32.875862
Data age (minutes): 10.08
SUCCESS: Silver data freshness validation passed!


In [0]:

from pyspark.sql import functions as F

df_gold_kpis = df_latest.agg(
    F.sum("bikes_available").alias("total_bikes_available"),

    F.sum("ebikes_available").alias("total_ebikes_available"),

    F.sum("docks_available").alias("total_docks_available"),

    F.sum(
        F.when(F.col("is_renting") == True, 1)
         .otherwise(0)
    ).alias("active_stations")
).withColumn(
    "snapshot_timestamp",
    F.lit(latest_timestamp)
)

display(df_gold_kpis)


total_bikes_available,total_ebikes_available,total_docks_available,active_stations,snapshot_timestamp
31351,12953,32801,2443,2026-11-24T22:37:32.875Z


In [0]:

# Save the latest network KPIs to Gold

df_gold_kpis.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "workspace.gold.citibike_network_kpis"
    )

print("Gold KPI table updated successfully!")


Gold KPI table updated successfully!


In [0]:

# Read the saved Gold KPI table
df_verify_gold = spark.table(
    "workspace.gold.citibike_network_kpis"
)

# Count KPI records
gold_row_count = df_verify_gold.count()

print("Gold KPI rows:", gold_row_count)

# Stop if the row count is incorrect
if gold_row_count != 1:
    raise ValueError(
        f"Gold KPI validation failed: expected 1 row, found {gold_row_count}"
    )

print("SUCCESS: Gold KPI table verified!")


Gold KPI rows: 1
SUCCESS: Gold KPI table verified!


In [0]:

from pyspark.sql import functions as F

# Identify stations with less than 20% bike availability
df_low_availability = df_latest.filter(
    (F.col("availability_percentage") < 20) &
    (F.col("capacity") > 0) &
    (F.col("is_renting") == True)
)

# Sort stations by lowest availability
df_low_availability = df_low_availability.orderBy(
    F.col("availability_percentage").asc()
)

print(
    "Low availability stations:",
    df_low_availability.count()
)


Low availability stations: 824


In [0]:

# Save updated low availability records to Gold

df_low_availability.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "workspace.gold.citibike_low_availability"
    )

print("Low availability Gold table updated!")


Low availability Gold table updated!


In [0]:

from pyspark.sql import functions as F

# Read the saved low-availability Gold table
df_verify_low = spark.table(
    "workspace.gold.citibike_low_availability"
)

# Count saved stations
saved_count = df_verify_low.count()

# Count stations that violate our filtering rules
invalid_stations = df_verify_low.filter(
    F.col("availability_percentage").isNull() |
    (F.col("availability_percentage") >= 20) |
    (F.col("capacity") <= 0) |
    F.col("capacity").isNull() |
    (F.col("is_renting") != True) |
    F.col("is_renting").isNull()
).count()

print("Low availability stations:", saved_count)
print("Invalid station records:", invalid_stations)

if invalid_stations > 0:
    raise ValueError(
        "Invalid records found in Gold low availability table!"
    )

print("SUCCESS: Low availability Gold table verified!")


Low availability stations: 824
Invalid station records: 0
SUCCESS: Low availability Gold table verified!
